In [3]:
import os
print(os.listdir("../data/raw_videos"))

['test-clip.mp4']


In [10]:
import cv2
import os

def extract_frames(video_path, output_dir, frame_interval=5):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = 0
    saved_frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval == 0:
            timestamp_sec = frame_count / fps
            frame_filename = f"frame_{frame_count:05d}.jpg"
            frame_path = os.path.join(output_dir, frame_filename)
            cv2.imwrite(frame_path, frame)

            saved_frames.append({
                "frame_number": frame_count,
                "timestamp_sec": round(timestamp_sec, 2),
                "frame_path": frame_path
            })

        frame_count += 1

    cap.release()
    print(f"Extracted {len(saved_frames)} frames from {frame_count} total frames (fps={fps})")
    return saved_frames

frames = extract_frames(
    video_path="../data/raw_videos/test-clip.mp4",
    output_dir="../data/extracted_frames",
    frame_interval=5
)

Extracted 164 frames from 818 total frames (fps=24.0)


In [12]:
print(frames[:3])

[{'frame_number': 0, 'timestamp_sec': 0.0, 'frame_path': '../data/extracted_frames\\frame_00000.jpg'}, {'frame_number': 5, 'timestamp_sec': 0.21, 'frame_path': '../data/extracted_frames\\frame_00005.jpg'}, {'frame_number': 10, 'timestamp_sec': 0.42, 'frame_path': '../data/extracted_frames\\frame_00010.jpg'}]


In [24]:
import numpy as np

def simulate_gps_route(frames, start_coord, end_coord):
    n = len(frames)
    lats = np.linspace(start_coord[0], end_coord[0], n)
    lons = np.linspace(start_coord[1], end_coord[1], n)

    for i, frame in enumerate(frames):
        frame["gps"] = (float(round(lats[i], 6)), float(round(lons[i], 6)))
    return frames

# Example: a short simulated drive — replace with any real-ish coordinates near you
frames = simulate_gps_route(frames, start_coord=(13.0827, 80.2707), end_coord=(13.0900, 80.2750))

print(frames[:3])

[{'frame_number': 0, 'timestamp_sec': 0.0, 'frame_path': '../data/extracted_frames\\frame_00000.jpg', 'gps': (13.0827, 80.2707), 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 71}}, {'frame_number': 5, 'timestamp_sec': 0.21, 'frame_path': '../data/extracted_frames\\frame_00005.jpg', 'gps': (13.082745, 80.270726), 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 71}}, {'frame_number': 10, 'timestamp_sec': 0.42, 'frame_path': '../data/extracted_frames\\frame_00010.jpg', 'gps': (13.08279, 80.270753), 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 71}}]


In [25]:
import requests
import time

WEATHER_API_KEY = __import__("os").environ.get("OPENWEATHER_API_KEY", "")   # paste your OpenWeatherMap key
weather_cache = {}

def get_weather(lat, lon, cache_precision=3):
    cache_key = (round(lat, cache_precision), round(lon, cache_precision))
    if cache_key in weather_cache:
        return weather_cache[cache_key]

    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"lat": lat, "lon": lon, "appid": WEATHER_API_KEY, "units": "metric"}

    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        weather_info = {
            "condition": data["weather"][0]["main"],
            "description": data["weather"][0]["description"],
            "temp_c": data["main"]["temp"],
            "humidity": data["main"]["humidity"]
        }
    except Exception as e:
        print(f"Weather fetch failed for ({lat}, {lon}): {e}")
        weather_info = {"condition": "Unknown", "description": "N/A", "temp_c": None, "humidity": None}

    weather_cache[cache_key] = weather_info
    return weather_info

# Test on just ONE coordinate first before running on all frames
test_weather = get_weather(frames[0]["gps"][0], frames[0]["gps"][1])
print(test_weather)

{'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 72}


In [26]:
import requests

url = "https://api.openweathermap.org/data/2.5/weather"
params = {"lat": 13.0827, "lon": 80.2707, "appid": WEATHER_API_KEY, "units": "metric"}

response = requests.get(url, params=params, timeout=5)
print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'coord': {'lon': 80.2707, 'lat': 13.0827}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 32.21, 'feels_like': 39.21, 'temp_min': 31.01, 'temp_max': 32.8, 'pressure': 1007, 'humidity': 72, 'sea_level': 1007, 'grnd_level': 1005}, 'visibility': 10000, 'wind': {'speed': 3.09, 'deg': 260}, 'clouds': {'all': 99}, 'dt': 1788510708, 'sys': {'type': 2, 'id': 2104103, 'country': 'IN', 'sunrise': 1788481683, 'sunset': 1788526114}, 'timezone': 19800, 'id': 1465730, 'name': 'Park Town', 'cod': 200}


In [27]:
for frame in frames:
    lat, lon = frame["gps"]
    frame["weather"] = get_weather(lat, lon)
    time.sleep(1)

print(f"Weather tagged for {len(frames)} frames")
print(frames[:3])

Weather tagged for 164 frames
[{'frame_number': 0, 'timestamp_sec': 0.0, 'frame_path': '../data/extracted_frames\\frame_00000.jpg', 'gps': (13.0827, 80.2707), 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 72}}, {'frame_number': 5, 'timestamp_sec': 0.21, 'frame_path': '../data/extracted_frames\\frame_00005.jpg', 'gps': (13.082745, 80.270726), 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 72}}, {'frame_number': 10, 'timestamp_sec': 0.42, 'frame_path': '../data/extracted_frames\\frame_00010.jpg', 'gps': (13.08279, 80.270753), 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 72}}]


In [28]:
import json

with open("../outputs/incidents/stage1_frames.json", "w") as f:
    json.dump(frames, f, indent=2)

print("Saved", len(frames), "frame records to stage1_frames.json")

Saved 164 frame records to stage1_frames.json
